# 00 Vorbereitung: Daten laden und ablegen

Dieses Notebook lädt den California-Housing-Datensatz über scikit-learn, speichert ihn als CSV-Datei und legt eine SQLite-Tabelle an.
Führe es einmal von oben nach unten aus, bevor du mit der Analyse beginnst. Beim ersten Mal brauchst du eine Internetverbindung.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent  # Projektwurzel, das Notebook liegt in notebooks/
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
CSV_PFAD = RAW_DIR / "california_housing.csv"
DB_PFAD = INTERIM_DIR / "projekt.sqlite"
TABELLE = "blockgruppen"

ERWARTETE_FORM = (20640, 9)
ERWARTETE_SPALTEN = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population",
                     "AveOccup", "Latitude", "Longitude", "MedHouseVal"]

assert (ROOT / "data").exists(), "Starte das Notebook aus dem Ordner notebooks/ des Projekts."
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
print("Projektwurzel:", ROOT.name)

Projektwurzel: G4_P1_CalHouse


## Daten laden

Liegt `data/raw/california_housing.csv` schon vor, wird die Datei gelesen. Sonst lädt scikit-learn den Datensatz aus dem Internet und speichert ihn dort.

In [2]:
HILFE_LADEN = (
    "Der Datensatz konnte nicht geladen werden.\n"
    "Was du tun kannst:\n"
    "1. Internetverbindung und Proxy prüfen und diese Zelle erneut ausführen. scikit-learn lädt die Daten beim ersten Aufruf "
    "aus dem Internet und legt sie im Ordner scikit_learn_data in deinem Benutzerverzeichnis ab.\n"
    "2. Klappt es dann noch nicht: den Dozenten holen und ihm diese Meldung zeigen."
)

if CSV_PFAD.exists():
    df = pd.read_csv(CSV_PFAD)
    print("CSV-Datei vorhanden und gelesen:", CSV_PFAD.relative_to(ROOT))
else:
    try:
        from sklearn.datasets import fetch_california_housing
        df = fetch_california_housing(as_frame=True).frame
    except Exception as fehler:
        raise RuntimeError(HILFE_LADEN) from fehler
    df.to_csv(CSV_PFAD, index=False)
    print("Neu geladen und gespeichert:", CSV_PFAD.relative_to(ROOT))

CSV-Datei vorhanden und gelesen: data\raw\california_housing.csv


## Form und Spalten prüfen

In [3]:
print("Form:", df.shape)
print("Spalten:", list(df.columns))
print(df.dtypes.astype(str).to_dict())

assert df.shape == ERWARTETE_FORM, f"Unerwartete Form {df.shape}, erwartet {ERWARTETE_FORM}."
assert list(df.columns) == ERWARTETE_SPALTEN, "Unerwartete Spalten. Wurde die richtige Datei geladen?"
df.head()

Form: (20640, 9)
Spalten: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']
{'MedInc': 'float64', 'HouseAge': 'float64', 'AveRooms': 'float64', 'AveBedrms': 'float64', 'Population': 'float64', 'AveOccup': 'float64', 'Latitude': 'float64', 'Longitude': 'float64', 'MedHouseVal': 'float64'}


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## SQLite-Tabelle anlegen

Die Analyse-Notebooks lesen die Daten aus dieser Tabelle. Die Datei `data/interim/projekt.sqlite` wird nicht eingecheckt.

In [4]:
conn = sqlite3.connect(DB_PFAD)
try:
    df.to_sql(TABELLE, conn, if_exists="replace", index=False)
    anzahl = pd.read_sql(f"SELECT COUNT(*) AS anzahl FROM {TABELLE}", conn)["anzahl"].iloc[0]
    kontrolle = pd.read_sql(f"SELECT * FROM {TABELLE} LIMIT 3", conn)
finally:
    conn.close()

assert anzahl == len(df), "Anzahl der Zeilen in SQLite weicht von der CSV-Datei ab."
print(f"Tabelle '{TABELLE}' in {DB_PFAD.name}: {anzahl} Zeilen")
kontrolle

Tabelle 'blockgruppen' in projekt.sqlite: 20640 Zeilen


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521


## Fertig

Die Daten liegen als CSV-Datei in `data/raw/` und als SQLite-Tabelle in `data/interim/`. Weiter geht es mit deinem ersten Notebook `01_data_understanding.ipynb`.